# 🕸️ RDF, RDFS & OWL Exercises
**Foundations of Artificial Intelligence · SS 2026 · University of Bremen**

---

This notebook covers the **programming part** (Exercise 6) of the RDF/OWL worksheet.  
Written tasks (✏️) are in `rdf_owl_exercises.md`.

You will use **`rdflib`** — Python's standard library for working with RDF graphs.

Run a cell with **Shift + Enter**.

---
## Setup – Install & Import rdflib

`rdflib` lets you build, query, and serialize RDF graphs in Python.  
The key building blocks are:

| Class | What it represents |
|---|---|
| `URIRef` | A URI — identifies a resource (`ex:Alice`) |
| `Literal` | A data value — a string, number, date (`"Alice"`, `30`) |
| `BNode` | A blank node — anonymous resource |
| `Namespace` | A shorthand prefix (`EX = Namespace("http://example.org/")`) |
| `Graph` | A collection of `(subject, predicate, object)` triples |

Adding a triple looks like this:
```python
g.add((EX.Alice, RDF.type, EX.Person))
#      subject    predicate  object
```

In [ ]:
# Install rdflib if needed (safe to re-run)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "rdflib", "-q"], check=True)
print("rdflib ready ✅")

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, OWL, URIRef, BNode
from rdflib.namespace import RDFS, XSD

# Our custom namespace — all our URIs start with this
EX = Namespace("http://example.org/")

print("Imports OK ✅")
print(f"Example URI:  {EX.Alice}")
print(f"Example type: {RDF.type}")

---
## Exercise 6 – Building an RDF Graph with rdflib 🐍

We will model a small **university knowledge base** with the following facts:

- `:Alice` is a `:Person`
- `:Alice` `:knows` `:Bob`
- `:Bob` is a `:Person`
- `:Alice` `:hasAge` `30` *(integer)*
- `:Bob` `:worksAt` `:UniBremen`
- `:UniBremen` is a `:University`

### 6.1 – Build the graph

In [ ]:
# Create a fresh graph and register our prefix
g = Graph()
g.bind("ex", EX)

# ── Already done for you: Alice is a Person ──────────────────────────────────
g.add((EX.Alice, RDF.type, EX.Person))

# YOUR CODE HERE ──────────────────────────────────────────────────────────────
# Add the remaining 5 triples:
#   1. EX.Alice  knows       EX.Bob
#   2. EX.Bob    rdf:type    EX.Person
#   3. EX.Alice  hasAge      Literal(30, datatype=XSD.integer)
#   4. EX.Bob    worksAt     EX.UniBremen
#   5. EX.UniBremen rdf:type EX.University
#
# Pattern: g.add((subject, predicate, object))



# ── Print the graph in Turtle format ─────────────────────────────────────────
print(g.serialize(format="turtle"))
print(f"\nTotal triples in graph: {len(g)}   ← expected 6")

### 6.2 – Query the graph

`g.triples((subject, predicate, object))` lets you search the graph.  
Use `None` as a wildcard for any position:

```python
# Find all triples where the predicate is rdf:type
for s, p, o in g.triples((None, RDF.type, None)):
    print(s, "is a", o)
```

In [ ]:
# ── 6.2a: Print all instances of EX.Person ───────────────────────────────────
print("=== All Persons ===")
for subj, pred, obj in g.triples((None, RDF.type, EX.Person)):
    print(" -", subj)

print()

# ── 6.2b: YOUR CODE HERE ─────────────────────────────────────────────────────
# Print everything that Alice (EX.Alice) is connected to.
# Hint: use g.triples((EX.Alice, None, None))
# For each triple, print:  predicate  →  object
print("=== All facts about Alice ===")


In [ ]:
# ── 6.2c: YOUR CODE HERE ─────────────────────────────────────────────────────
# Retrieve Alice's age from the graph and print it.
# Hint: iterate g.triples((EX.Alice, EX.hasAge, None))
# The object will be a Literal — use int(obj) to convert it.

print("=== Alice's age ===")


### 6.3 – Serialize & Reload (Round-trip)

A key feature of RDF is that graphs can be saved to files and loaded by any RDF-aware tool.  
We'll save our graph to **Turtle** format (`.ttl`) and reload it to verify nothing was lost.

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
g.serialize(destination="knowledge_base.ttl", format="turtle")
print("Saved to knowledge_base.ttl ✅")

# ── Reload ───────────────────────────────────────────────────────────────────
g2 = Graph()
g2.parse("knowledge_base.ttl", format="turtle")

# ── Verify ───────────────────────────────────────────────────────────────────
# YOUR CODE HERE:
# Check that g2 has the same number of triples as g.
# Print a success or failure message.
# Hint: use len(g) and len(g2)


---
## Bonus – Exercise 6.4: Extending the Graph with RDFS

So far our graph has no **class hierarchy**. Let's add RDFS schema information:

- A `:University` is a subclass of `:Organisation`
- A `:Person` is a subclass of `:Agent`
- `:knows` has domain `:Person` and range `:Person`
- `:worksAt` has domain `:Person` and range `:Organisation`

Then we will write a small **RDFS reasoner** that applies `rdfs:subClassOf` to infer new `rdf:type` triples.

In [ ]:
# ── Add RDFS schema triples to the existing graph g ──────────────────────────

# Subclass hierarchy
g.add((EX.University, RDFS.subClassOf, EX.Organisation))
g.add((EX.Person,     RDFS.subClassOf, EX.Agent))

# Domain and range for :knows
g.add((EX.knows, RDFS.domain, EX.Person))
g.add((EX.knows, RDFS.range,  EX.Person))

# YOUR CODE HERE:
# Add domain and range for :worksAt
#   domain: EX.Person
#   range:  EX.Organisation


print(f"Graph now has {len(g)} triples (including schema)")

In [ ]:
# ── Mini RDFS reasoner: apply rdfs:subClassOf ─────────────────────────────────
#
# Rule: if  X rdf:type C  and  C rdfs:subClassOf D
#       then infer  X rdf:type D
#
# We repeat until no new triples are added (fixpoint).

def rdfs_subclass_closure(graph):
    """
    Apply rdfs:subClassOf entailment until no new rdf:type triples can be derived.
    Adds inferred triples directly to `graph`.
    Returns the number of new triples added.
    """
    new_triples = []

    # Find all subClassOf relationships
    subclass_pairs = list(graph.triples((None, RDFS.subClassOf, None)))

    for subclass, _, superclass in subclass_pairs:
        # Find all individuals of the subclass
        for individual, _, _ in graph.triples((None, RDF.type, subclass)):

            # YOUR CODE HERE:
            # If the triple (individual, RDF.type, superclass) is NOT yet in the graph,
            # append it to new_triples.
            # Hint: use  (individual, RDF.type, superclass) not in graph
            pass

    for triple in new_triples:
        graph.add(triple)

    return len(new_triples)


# Run the reasoner to fixpoint
total_inferred = 0
while True:
    added = rdfs_subclass_closure(g)
    total_inferred += added
    if added == 0:
        break

print(f"Reasoner finished. Inferred {total_inferred} new triple(s).")
print()

# Print all rdf:type triples to see what was inferred
print("=== All rdf:type assertions (including inferred) ===")
for s, p, o in sorted(g.triples((None, RDF.type, None)), key=lambda t: str(t[0])):
    # Skip OWL/RDF meta-triples
    if not str(o).startswith("http://www.w3.org"):
        print(f"  {s.split('/')[-1]:15}  rdf:type  {o.split('/')[-1]}")

**✏️ Reflection** – Which new `rdf:type` triples did the reasoner infer?  
Why does RDFS need a reasoner to make these conclusions — can a plain database do the same thing?

> *Your answer here...*

---
## Bonus – Exercise 6.5: Exploring a Real-World Ontology

Let's load a small slice of a real ontology — the **FOAF** (Friend of a Friend) vocabulary —
and inspect its class hierarchy.

FOAF is used across the web to describe people and their social connections.

In [ ]:
# ── Load FOAF from the web ────────────────────────────────────────────────────
foaf_graph = Graph()
foaf_graph.parse("http://xmlns.com/foaf/spec/20140114.rdf", format="xml")
print(f"FOAF ontology loaded: {len(foaf_graph)} triples")
print()

# ── Print all rdfs:subClassOf relations in FOAF ───────────────────────────────
print("=== FOAF subClassOf hierarchy ===")
for sub, _, sup in foaf_graph.triples((None, RDFS.subClassOf, None)):
    # Shorten URIs for readability
    sub_short = str(sub).split("/")[-1].split("#")[-1]
    sup_short = str(sup).split("/")[-1].split("#")[-1]
    print(f"  {sub_short}  ⊆  {sup_short}")

print()

# YOUR CODE HERE:
# Count how many owl:ObjectProperty and owl:DatatypeProperty entries FOAF defines.
# Hint: use foaf_graph.triples((None, RDF.type, OWL.ObjectProperty)) etc.
print("=== FOAF property counts ===")


---
## 📊 Summary

Run the cell below for a recap of everything you built.

In [ ]:
print("What you did in this notebook:")
print()
print("  6.1  Built an RDF graph from scratch using rdflib")
print("       URIRefs, Literals, Namespaces, g.add()")
print()
print("  6.2  Queried the graph with g.triples() and wildcard None")
print()
print("  6.3  Serialized to Turtle (.ttl) and reloaded (round-trip)")
print()
print("  6.4  Added RDFS schema (subClassOf, domain, range)")
print("       and implemented a mini RDFS reasoner")
print()
print("  6.5  Loaded and inspected a real-world ontology (FOAF)")
print()
print("These are the practical building blocks of the Semantic Web! 🎓")

---
🎓 **Well done!** Written tasks (Parts 1–5) are in `rdf_owl_exercises.md`.